# 04 - Analytics & Prediction: Fare vs Reliability Regression Models

Compares 3 regression models predicting `reliability_score` from fare and service-volume features, per the brief's Analytics & Prediction requirement (Section 3).

**Run this AFTER `03_Join_Reliability_Supabase.ipynb`** -- it re-derives the same merged route-level table used there.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Rebuild the route-level merged table

Same aggregation and join logic as Notebook 03, plus two extra features (`fare_std`, `fare_product_count`) to give the models more to work with.

In [ ]:
tt = pd.read_csv(r'D:\Dataset\timetables_flat.csv', low_memory=False)
fr = pd.read_csv(r'D:\Dataset\fares_flat.csv')
di = pd.read_csv(r'D:\Dataset\catalogue\disruptions_data_catalogue.csv')
di = di[di['Organisation'] == 'West of England']

fares_agg = fr.groupby('route').agg(
    avg_fare=('price_gbp', 'mean'),
    fare_std=('price_gbp', 'std'),
    fare_product_count=('price_gbp', 'count')
).reset_index()
fares_agg['route'] = fares_agg['route'].astype(str)
fares_agg['fare_std'] = fares_agg['fare_std'].fillna(0)

tt_agg = (tt.groupby('line_name')
          .agg(trip_count=('vehicle_journey_code', 'nunique'),
               stop_count=('stop_point_ref', 'nunique'))
          .reset_index()
          .rename(columns={'line_name': 'route'}))
tt_agg['route'] = tt_agg['route'].astype(str)

di_agg = di.groupby('Services affected').size().reset_index(name='disruption_count')
di_agg = di_agg.rename(columns={'Services affected': 'route'})
di_agg['route'] = di_agg['route'].astype(float).astype(int).astype(str)

merged = tt_agg.merge(fares_agg, on='route', how='inner').merge(di_agg, on='route', how='left')
merged['disruption_count'] = merged['disruption_count'].fillna(0)
merged['reliability_score'] = merged['disruption_count'] / merged['trip_count']

print('Total routes for modelling:', len(merged))
merged

## 3. A note on dataset size before modelling

This dataset has only **25 rows** (one per bus route) -- far smaller than typical ML datasets. This is expected given the scope of a single operator (First Bus) in one city (Bristol): there are only so many distinct routes.

This is being documented transparently here and should be carried into the report's **Critical Reflection** section: with n=25, any model's results should be treated as exploratory/illustrative rather than a statistically robust predictive tool. The large-scale "big data" component of this project lies in the underlying Timetables data (500k+ rows, processed via PySpark) -- this final modelling step operates on the *aggregated* route-level summary, which is necessarily small since there are only 25 routes to summarise.

## 4. Train/test split

In [ ]:
feature_cols = ['avg_fare', 'fare_std', 'fare_product_count', 'trip_count', 'stop_count']
X = merged[feature_cols]
y = merged['reliability_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('Train size:', len(X_train), '| Test size:', len(X_test))

## 5. Compare 3 regression models

Per the brief: Linear Regression (simple baseline), Random Forest (ensemble, handles non-linearity), Gradient Boosting (sequential ensemble, often strongest on small tabular data).

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    predictions[name] = preds

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

results_df = pd.DataFrame(results)
results_df

## 6. Interpreting the results

If R² is negative for all models (likely, given n=25 and near-flat fares), this means each model performs *worse* than simply predicting the average reliability score every time. This is a valid and expected finding here -- it directly demonstrates, quantitatively, that **fare has little to no predictive relationship with reliability for this operator's route network**, which is itself a legitimate answer to the project's research question, not a failure of the modelling process.

For the report's Results section: state the RMSE/MAE/R2 table, then explain that the near-uniform fare structure (most routes priced at GBP1.80) limits the available signal -- this is a data limitation, not a methodological one.

## 7. Feature importance (Random Forest) -- which factor matters most?

In [ ]:
rf_model = models['Random Forest']
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

importances

## 8. Save results for the report

In [ ]:
results_df.to_csv(r'D:\Dataset\output\ml_model_comparison.csv', index=False)
importances.to_csv(r'D:\Dataset\output\ml_feature_importance.csv', index=False)
print('Saved ml_model_comparison.csv and ml_feature_importance.csv to D:\\Dataset\\output')